**Sirroz bemorlarning omon qolish prognozini aniqlash**

**Kutubxonalar**

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,RandomizedSearchCV
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, log_loss

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier

**DataSet**

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/multiclassificationtask/train.csv")
test = pd.read_csv("/kaggle/input/competitions/multiclassificationtask/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/multiclassificationtask/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

In [ ]:
print(train.describe())

print(train.info())

In [ ]:
# Noto‘g‘ri classni olib tashlash

train = train[train["Status"] != 3].reset_index(drop=True)

**Target Encoding**

In [ ]:
le_target = LabelEncoder()

train["Status"] = le_target.fit_transform(train["Status"])

**Vizualizatsiya**

In [ ]:
# Target distribution

plt.figure(figsize=(6,4))
sns.countplot(data=train, x="Status")
plt.title("Target Distribution")
plt.show()

In [ ]:
# Missing Values

missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(10,5))
missing.plot(kind='bar')
plt.title("Missing Values")
plt.show()

In [ ]:
# Numeric feature distribution

numeric_cols = train.select_dtypes(include=['int64','float64']).columns

train[numeric_cols].hist(
    figsize=(15,10),
    bins=30
)

plt.show()

In [ ]:
# Correlation Matrix

plt.figure(figsize=(12,8))

sns.heatmap(
    train[numeric_cols].corr(),
    cmap="coolwarm"
)

plt.title("Correlation Matrix")
plt.show()

**X/Y**

In [ ]:
X = train.drop(["Status","id"], axis=1)
y = train["Status"]
X_test = test.drop(["id"], axis=1)

In [ ]:
# Numeric & categorical columns

numeric_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include='object').columns

# Numeric imputer

num_imputer = SimpleImputer(strategy='median')
X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

# Categorical imputer

cat_imputer = SimpleImputer(strategy='most_frequent')
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

# Label Encoding

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

**Cross Validation**

In [ ]:
kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

**Models**

RandomForest

In [ ]:
rf_scores = []

model_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    random_state=42
)

for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx] 
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_rf.fit(X_train, y_train)
    val_pred = model_rf.predict_proba(X_val)

    score = log_loss(y_val, val_pred, labels=model_rf.classes_)
    rf_scores.append(score)

print("RandomForest LogLoss:", np.mean(rf_scores))

XGB

In [ ]:
xgb_scores = []

model_xgb = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx] 
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_xgb.fit(X_train, y_train)
    val_pred = model_xgb.predict_proba(X_val)

    score = log_loss(y_val, val_pred, labels=model_xgb.classes_)
    xgb_scores.append(score)

print("XGBoost LogLoss:", np.mean(xgb_scores))

Catboost

In [ ]:
cat_scores = []

model_cat = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    loss_function="MultiClass",
    verbose=False,
    random_state=42
)

for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_cat.fit(X_train, y_train)

    val_pred = model_cat.predict_proba(X_val)

    score = log_loss(y_val, val_pred, labels=model_cat.classes_)
    cat_scores.append(score)

print("CatBoost LogLoss:", np.mean(cat_scores))

GradientBoosting

In [ ]:
gb_scores = []

model_gb = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3
)

for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_gb.fit(X_train, y_train)

    val_pred = model_gb.predict_proba(X_val)

    score = log_loss(y_val, val_pred, labels=model_gb.classes_)
    gb_scores.append(score)


print("GradientBoosting LogLoss:", np.mean(gb_scores))

HistGradientBoosting

In [ ]:
kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

hist_scores = []

model_hist = HistGradientBoostingClassifier(
    max_iter=600,
    learning_rate=0.03,
    max_depth=6
)

for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_hist.fit(X_train, y_train)

    val_pred = model_hist.predict_proba(X_val)

    score = log_loss(y_val, val_pred, labels=model_hist.classes_)
    hist_scores.append(score)

print("HistGradientBoosting LogLoss:", np.mean(hist_scores))

**Natijalar**

In [ ]:
models = [
    "CatBoost",
    "XGBoost",
    "RandomForest",
    "GradientBoost",
    "HistGradientBoost"
]

scores = [
    np.mean(cat_scores),
    np.mean(xgb_scores),
    np.mean(rf_scores),
    np.mean(gb_scores),
    np.mean(hist_scores),
]

Grafik

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    x=models,
    y=scores
)

plt.title("Model Performance Comparison")
plt.ylabel("LogLoss")
plt.xticks(rotation=45)

plt.show()

**Eng yaxshi model**

In [ ]:
X_filtered = X[y.isin([0,1,2])]
y_filtered = y[y.isin([0,1,2])]

kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) 

param_grid = {
    "n_estimators": [100, 200],      
    "max_depth": [6, 8, 10],         
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf = RandomForestClassifier(random_state=42)

rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,                
    cv=kf,
    scoring="neg_log_loss",
    n_jobs=-1,
    random_state=42
)

rand_search.fit(X_filtered, y_filtered)

print("Best Params:", rand_search.best_params_)
print("Best Score:", -rand_search.best_score_)

best_rf = rand_search.best_estimator_
best_rf.fit(X_filtered, y_filtered)

test_pred = best_rf.predict_proba(X_test)

**Submission**

In [ ]:
submission = pd.DataFrame()
submission['id'] = test['id']
submission['Status_C'] = test_pred[:, 0]
submission['Status_CL'] = test_pred[:, 1]
submission['Status_D'] = test_pred[:, 2]

submission.to_csv('submission.csv', index=False)